# Preprocesamiento y PCA - Riesgo Geológico Global

Pipeline modular con **sklearn Pipeline + ColumnTransformer**, exportable a producción con `joblib`.

**Autor:** Juan (Persona 4 - Dúo B)

In [ ]:
import sys, os, warnings
import numpy as np
import pandas as pd

project_root = os.path.join(os.getcwd(), "..")
if os.path.basename(os.getcwd()) == "nb":
    sys.path.insert(0, project_root)
else:
    sys.path.insert(0, os.getcwd())

warnings.filterwarnings("ignore")

from src.data_loader import cargar_datos_combinados
from src.preprocessing import pipeline_preprocesamiento_pca, cargar_pipeline
from src.visualization import (
    plot_varianza_acumulada,
    plot_pca_2d,
    plot_biplot,
    plot_pca_interactivo,
)
print("Librerias cargadas correctamente")

## 1. Carga de datos combinados

In [ ]:
df_raw = cargar_datos_combinados()
print(f"Dimensiones: {df_raw.shape}")
print(f"Columnas: {list(df_raw.columns)}")
df_raw.head(3)

## 2. Pipeline de Preprocesamiento + PCA

La función `pipeline_preprocesamiento_pca()` ejecuta:
- Transformación `log1p` en columnas con skew > 0.75
- One-Hot Encoding en variables categóricas
- `StandardScaler` (vía `ColumnTransformer`)
- `PCA` con selección automática al 85% de varianza
- Exporta el pipeline completo con `joblib`

In [ ]:
df_pca, pipeline, df_scaled = pipeline_preprocesamiento_pca(
    df_raw,
    target_variance=0.85,
    save_path="models/pipeline_riesgo.pkl",
)
print(f"PCA resultante: {df_pca.shape[1]} componentes")
print(f"Varianza explicada acumulada: {pipeline.named_steps['pca'].explained_variance_ratio_.cumsum()[-1]:.2%}")
df_pca.head()

## 3. Varianza Explicada Acumulada

In [ ]:
plot_varianza_acumulada(
    pipeline.named_steps["pca"],
    threshold=0.85,
    save_path="figures/varianza_acumulada.png",
);

## 4. Proyección 2D (PC1 vs PC2)

In [ ]:
plot_pca_2d(
    df_pca,
    pca_model=pipeline.named_steps["pca"],
    save_path="figures/pca_2d.png",
);

## 5. Biplot

Las flechas indican la contribución de cada variable original a los componentes.

In [ ]:
pca = pipeline.named_steps["pca"]
feature_names = df_scaled.columns.tolist()

plot_biplot(
    df_pca,
    pca_model=pca,
    feature_names=feature_names,
    save_path="figures/biplot.png",
);

## 6. Gráfico Interactivo (Plotly)

In [ ]:
plot_pca_interactivo(
    df_pca,
    pca_model=pca,
    save_path="figures/pca_interactivo.html",
);
print("HTML interactivo guardado en figures/pca_interactivo.html")

## 7. Carga del Pipeline Exportado

Verificación de que el pipeline se puede reutilizar en producción.

In [ ]:
loaded = cargar_pipeline("models/pipeline_riesgo.pkl")
sample = df_raw.head(5)
X_transformed = loaded.transform(sample)
print(f"Shape del transform: {X_transformed.shape}")
print("Pipeline cargado y funcional.")

---
*Notebook generado automáticamente. Los resultados dependen de los datos de entrada.*